# 00 - Preparación / compilación de bases EPH

**Este es el notebook base del proyecto.** Compila las bases de la EPH descargadas
manualmente del INDEC y subidas a **Google Drive** (carpeta `carga_EPH`): une la base de
**individuos** con la de **hogares** por el código de vínculo (`CODUSU` + `NRO_HOGAR`) y
deja los datasets listos en `data/processed/` para que **el resto de los notebooks
(01-05) los procesen** sin tener que volver a descargar ni unir las bases.

**Salida (en `data/processed/`):**
- `eph_T<Q><YY>.parquet`: **un archivo por trimestre**, ya con individuos+hogares unidos
  y columnas `ANIO`/`TRIMESTRE`. No se genera un único panel combinado: desbordaría la
  RAM de Colab y el límite de 100 MB de GitHub. Los notebooks leen lo que necesitan con
  `load_panel(columns=..., quarters=...)`.

**Flujo:** Setup (clonar repo + montar Drive) → Diagnóstico de `carga_EPH` →
Trimestres disponibles → Compilar (1 parquet por trimestre) → Verificación (merge +
quiebre de esquema 4T2023).

**Cómo agregar un trimestre nuevo:**
1. Descargar del INDEC el `.zip` del trimestre (ej. `EPH_usu_4_Trim_2025_txt.zip`),
   que contiene `usu_individual_T<Q><YY>.txt` y `usu_hogar_T<Q><YY>.txt`.
2. Subir el `.zip` (sin descomprimir) a Google Drive, carpeta `carga_EPH`.
3. Volver a correr este notebook: detecta automáticamente todos los trimestres
   presentes en `carga_EPH` (no hace falta editar ninguna lista).

> ⚠️ **Quiebre de esquema en 4T2023.** Los trimestres ≤ T3-2023 tienen menos columnas
> (177 individual / 88 hogar) que los ≥ T4-2023 (235 / 98). Las variables nuevas
> (`EMPLEO`, `SECTOR`, ingresos/pensiones desagregados, deciles `P_DECCF`) no existen en
> los trimestres viejos. Ver `.claude/memoria_EPH.md`.

## Setup (Colab)

Clona el repo para tener acceso a `src/data_loader.py`, y monta Google Drive para
leer los `.zip` subidos a `carga_EPH`.

In [ ]:
import sys, os

REPO_URL = "https://github.com/santiagoriverti/analisis_EPH.git"
REPO_DIR = "/content/analisis_EPH"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

## Diagnóstico: ¿qué hay en carga_EPH?

Si la celda "Disponibles" más abajo devuelve una lista vacía, correr esta celda
primero para ver qué archivos detecta Colab en la carpeta de Drive y qué contiene
un `.zip` de ejemplo.

In [ ]:
from src.data_loader import DRIVE_DIR
import os, zipfile

print("DRIVE_DIR:", DRIVE_DIR, "| existe:", os.path.isdir(DRIVE_DIR))

if os.path.isdir(DRIVE_DIR):
    archivos = os.listdir(DRIVE_DIR)
    print(f"Archivos encontrados ({len(archivos)}):")
    for a in archivos[:10]:
        print(" -", a)

    zips = [a for a in archivos if a.lower().endswith(".zip")]
    if zips:
        ejemplo = os.path.join(DRIVE_DIR, zips[0])
        print(f"\nContenido de {zips[0]}:")
        with zipfile.ZipFile(ejemplo) as zf:
            for n in zf.namelist():
                print(" -", n)

## Chequeo de disponibilidad

Escanea `/content/drive/MyDrive/carga_EPH` (y `data/raw/` del repo) buscando los `.zip`
del INDEC, y lista los trimestres detectados (con ambas bases, individual y hogar).

In [ ]:
from src.data_loader import list_available_quarters, _period_tag

available = list_available_quarters()
print("Disponibles:", [_period_tag(y, p) for y, p in available])

## Compilación del panel (memory-safe)

Procesa **un trimestre por vez**: une individuos + hogares por `CODUSU` + `NRO_HOGAR`,
agrega `ANIO`/`TRIMESTRE` y guarda **un parquet por trimestre** en `data/processed/`.

> No se arma un único DataFrame con los 36 trimestres en memoria (desbordaría la RAM de
> Colab), ni un único `eph_panel.parquet` (superaría el límite de 100 MB de GitHub). Los
> notebooks 01-05 leen los trimestres que necesiten con `load_panel(columns=..., quarters=...)`.

In [ ]:
from src.data_loader import build_panel
import pandas as pd

resumen = build_panel(available)
df_resumen = pd.DataFrame(resumen)
print("Trimestres compilados:", len(df_resumen))
print("Total de filas:", df_resumen["filas"].sum())
df_resumen[["year", "period", "filas", "columnas"]]

In [ ]:
# Verificación del merge: muestra de un trimestre (lee solo columnas clave, memory-safe)
from src.data_loader import load_panel

muestra = load_panel(
    columns=["CODUSU", "NRO_HOGAR", "COMPONENTE", "CH04", "CH06", "ESTADO", "ITF", "IPCF"],
    quarters=[(2025, 4)],
)
print("Shape muestra T4-2025:", muestra.shape)
muestra.head()

## Cómo usan estos datos los notebooks 01-05

Cada notebook de análisis arranca clonando el repo y llamando a `load_panel`, pidiendo
solo las columnas (y trimestres) que necesite para no cargar todo en memoria:

```python
from src.data_loader import load_panel

# Ejemplo (demografía): edad, sexo, región y ponderador, todos los trimestres
df = load_panel(columns=["CH06", "CH04", "REGION", "PONDERA"])

# Ejemplo (informalidad, solo esquema nuevo): restringir a T4-2023 en adelante
df = load_panel(
    columns=["EMPLEO", "SECTOR", "PONDERA"],
    quarters=[(2023, 4), (2024, 1), (2024, 2), (2024, 3), (2024, 4)],
)
```

Los parquets por trimestre viven en `data/processed/`. Si se quieren versionar en GitHub
para que los notebooks los lean por `raw.githubusercontent.com`, commitearlos desde el
entorno local (cada archivo trimestral entra holgado en el límite de 100 MB de GitHub).

In [ ]:
# Variables del esquema "nuevo" (desde 4T2023): primer trimestre con datos no nulos.
# Lee solo esas columnas de todos los trimestres (memory-safe).
vars_nuevas = ["EMPLEO", "SECTOR", "P_DECCF", "V2_01_M", "V5_01_M"]

chk = load_panel(columns=vars_nuevas)

for col in vars_nuevas:
    if col not in chk.columns:
        print(f"{col}: no está en ningún trimestre")
        continue
    con_datos = chk.loc[chk[col].notna(), ["ANIO", "TRIMESTRE"]]
    if con_datos.empty:
        print(f"{col}: sin datos en ningún trimestre")
    else:
        primer = con_datos.sort_values(["ANIO", "TRIMESTRE"]).iloc[0]
        print(f"{col}: primer trimestre con datos -> {int(primer.ANIO)}T{int(primer.TRIMESTRE)}")

## Subir resultados a GitHub (opcional)

Los archivos `.parquet` generados en `data/processed/` quedan disponibles para los demás notebooks vía `raw.githubusercontent.com` una vez que se comiteen y pusheen al repo (esto se hace desde el entorno local, no desde Colab).